In [1]:
# Core RAG Frameworks
!pip install langchain
!pip install langchain-openai
!pip install langchain-community
!pip install langchain-text-splitters
!pip install langchain-experimental
!pip install langgraph

# Vector Stores
!pip install chromadb
!pip install langchain-chroma

# Import the Python SDK for Colab secrets
from google.colab import userdata


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.2/122.2 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.7/561.7 kB 14.6 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.4.9
    Uninstalling langchain-core-1.4.9:
      Successfully uninstalled langchain-core-1.4.9
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 54.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.4 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-

In [5]:
import json
import os
from typing import List, Dict, Any
from langchain_core.tools import Tool
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage, SystemMessage
import google.colab

In [14]:
class SupportTicketTools:
    """Collection of tools for the support assistant agent."""

    def __init__(self, tickets_path: str = './sample_data/synthetic_tickets.json'):
        """
        Initialize ticket data + retrieval infrastructure once at startup.

        This keeps tool calls fast because embeddings/vector index are prepared
        ahead of time rather than rebuilt per request.
        """
        # Load tickets
        with open(tickets_path, 'r', encoding='utf-8') as f:
            self.tickets = json.load(f)

        # Fetch the API key from Colab secrets
        openai_api_key = userdata.get('OPENAI_APIKEY')

        # Create vector store for RAG
        self.embeddings = OpenAIEmbeddings(
            model=os.getenv('OPENAI_EMBEDDING_MODEL', 'text-embedding-3-small'),
            api_key=openai_api_key # Pass the fetched API key
        )
        self._setup_vectorstore()
    def _setup_vectorstore(self):
        """
        Build a semantic-search index from ticket text.

        Each ticket becomes one Document with metadata so agent responses can
        include both narrative context and structured filters/citations.
        """
        documents = []
        for ticket in self.tickets:
            content = f"""Ticket ID: {ticket['ticket_id']}
Title: {ticket['title']}
Description: {ticket['description']}
Resolution: {ticket['resolution']}
Category: {ticket['category']}
Priority: {ticket['priority']}"""

            doc = Document(
                page_content=content,
                metadata={
                    'ticket_id': ticket['ticket_id'],
                    'category': ticket['category'],
                    'priority': ticket['priority'],
                    'title': ticket['title']
                }
            )
            documents.append(doc)

        # Persist locally so repeated runs reuse indexed data on disk.
        self.vectorstore = Chroma.from_documents(
            documents=documents,
            embedding=self.embeddings,
            persist_directory="./agent_vectorstore"
        )

    def search_similar_tickets(self, query: str) -> str:
        """
        Search for similar support tickets using semantic similarity.

        Args:
            query: User's problem description or question

        Returns:
            Formatted string with relevant tickets
        """
        # Top-k semantic retrieval by embedding similarity.
        results = self.vectorstore.similarity_search(query, k=3)

        if not results:
            return "No similar tickets found."

        output = "Found similar tickets:\n\n"
        for i, doc in enumerate(results, 1):
            output += f"--- Ticket {i} ---\n"
            output += doc.page_content + "\n\n"

        return output

    def get_ticket_by_id(self, ticket_id: str) -> str:
        """
        Retrieve a specific ticket by its ID.

        Args:
            ticket_id: The ticket ID (e.g., TICK-001)

        Returns:
            Formatted ticket details or error message
        """
        # Normalize input to avoid case/whitespace mismatches.
        ticket_id = ticket_id.upper().strip()

        for ticket in self.tickets:
            if ticket['ticket_id'] == ticket_id:
                return f"""Ticket ID: {ticket['ticket_id']}
Title: {ticket['title']}
Description: {ticket['description']}
Resolution: {ticket['resolution']}
Category: {ticket['category']}
Priority: {ticket['priority']}
Created: {ticket['created_date']}
Resolved: {ticket['resolved_date']}"""

        return f"Ticket {ticket_id} not found in the database."

    def search_by_category(self, category: str) -> str:
        """
        Find all tickets in a specific category.

        Args:
            category: Category name (e.g., Authentication, Database, Payment)

        Returns:
            List of tickets in the category
        """
        # Normalize category string for case-insensitive exact matching.
        category = category.strip()
        matching = [t for t in self.tickets if t['category'].lower() == category.lower()]

        if not matching:
            available = list(set(t['category'] for t in self.tickets))
            return f"No tickets found in category '{category}'. Available categories: {', '.join(available)}"

        output = f"Found {len(matching)} tickets in category '{category}':\n\n"
        for ticket in matching:
            output += f"• [{ticket['ticket_id']}] {ticket['title']} (Priority: {ticket['priority']})\n"

        return output

    def get_ticket_statistics(self, input: str = "") -> str:
        """
        Get statistics and insights about the ticket database.

        Args:
            input: Optional parameter (not used, but required by Tool interface)

        Returns:
            Statistics about tickets
        """
        total = len(self.tickets)

        # Aggregate counts for dashboard/overview style questions.
        # Using dict accumulation keeps this dependency-free and explicit.
        categories = {}
        priorities = {}
        for ticket in self.tickets:
            categories[ticket['category']] = categories.get(ticket['category'], 0) + 1
            priorities[ticket['priority']] = priorities.get(ticket['priority'], 0) + 1

        output = f"Ticket Database Statistics:\n"
        output += f"Total Tickets: {total}\n\n"

        output += "By Category:\n"
        for cat, count in sorted(categories.items(), key=lambda x: x[1], reverse=True):
            output += f"  • {cat}: {count}\n"

        output += "\nBy Priority:\n"
        for pri, count in sorted(priorities.items(), key=lambda x: x[1], reverse=True):
            output += f"  • {pri}: {count}\n"

        return output

    def get_tools(self) -> List[Tool]:
        """
        Get all tools as LangChain Tool objects.

        Returns:
            List of Tool objects for the agent
        """
        # Tool descriptions are part of the model prompt context.
        # High-quality descriptions materially improve tool-selection accuracy.
        return [
            Tool(
                name="SearchSimilarTickets",
                func=self.search_similar_tickets,
                description="""Use this tool to search for similar support tickets based on a problem description or question.
Input should be a clear description of the issue or question.
This is the PRIMARY tool for answering "how to fix" or "similar issue" questions."""
            ),
            Tool(
                name="GetTicketByID",
                func=self.get_ticket_by_id,
                description="""Use this tool to retrieve details of a specific ticket by its ID.
Input should be a ticket ID like 'TICK-001'.
Use this when the user mentions a specific ticket number."""
            ),
            Tool(
                name="SearchByCategory",
                func=self.search_by_category,
                description="""Use this tool to find all tickets in a specific category.
Input should be a category name like 'Authentication', 'Database', 'Payment', 'Mobile', etc.
Use this when the user wants to see all issues of a certain type."""
            ),
            Tool(
                name="GetTicketStatistics",
                func=self.get_ticket_statistics,
                description="""Use this tool to get statistics about the ticket database including total count,
category distribution, and priority breakdown.
No input needed - just call this tool.
Use this when the user asks for statistics, summaries, or overview of tickets."""
            )
        ]

In [15]:
# ============================================================================
# PART 1: Setup Agent with Tools
# ============================================================================
print("\n" + "="*80)
print("PART 1: Setting Up the Agent")
print("="*80)

print("\nInitializing LLM...")

# Fetch the API key from Colab secrets
openai_api_key = userdata.get('OPENAI_APIKEY')

llm = ChatOpenAI(
    model=os.getenv('OPENAI_CHAT_MODEL', 'gpt-4o-mini'),
    temperature=0,
    api_key=openai_api_key # Pass the fetched API key
)
print("✓ LLM initialized")

print("\nCreating agent tools...")
tool_manager = SupportTicketTools()
tools = tool_manager.get_tools()

# Convert tools to OpenAI function format
tool_definitions = []
for tool in tools:
    tool_definitions.append({
        "type": "function",
        "function": {
            "name": tool.name,
            "description": tool.description,
            "parameters": {
                "type": "object",
                "properties": {
                    "input": {
                        "type": "string",
                        "description": "The input to the tool"
                    }
                },
                "required": ["input"]
            }
        }
    })

# Bind tools to LLM
llm_with_tools = llm.bind(tools=tool_definitions)

print(f"✓ Created {len(tools)} tools:")
for tool in tools:
    print(f"  • {tool.name}: {tool.description.split('.')[0]}")


PART 1: Setting Up the Agent

Initializing LLM...
✓ LLM initialized

Creating agent tools...
✓ Created 4 tools:
  • SearchSimilarTickets: Use this tool to search for similar support tickets based on a problem description or question
  • GetTicketByID: Use this tool to retrieve details of a specific ticket by its ID
  • SearchByCategory: Use this tool to find all tickets in a specific category
  • GetTicketStatistics: Use this tool to get statistics about the ticket database including total count,
category distribution, and priority breakdown


In [ ]:
# NOTE: LangChain also provides `bind_tools()` which does the same thing without
# needing the manual tool_definitions conversion above:
#
#   llm_with_tools = llm.bind_tools(tools)
#
# `bind_tools()` auto-converts LangChain Tool objects to the OpenAI function schema
# using the tool's name, description, and function signature. We use the manual
# approach here so you can see exactly what the OpenAI format looks like.
# If you switch to bind_tools(), also update line ~129: tool_call["args"] will use
# the actual function parameter names (e.g. "query") instead of the hardcoded "input".

In [16]:
def run_agent(query: str, max_iterations: int = 5) -> str:
    """
    Run a ReAct-style tool-calling loop until the model returns a final answer.

    Loop behavior:
    1) Model sees conversation + tool schema.
    2) Model either answers directly OR emits one/more tool calls.
    3) We execute each tool call, append ToolMessage results.
    4) Repeat until no tool calls remain or iteration cap is reached.
    """
    messages = [
        SystemMessage(content="""You are an expert support desk assistant that helps troubleshoot technical issues.

You have access to a database of previous support tickets with their resolutions.
Use your tools to find relevant information and provide helpful, accurate answers.

Guidelines:
- ALWAYS search for similar tickets when asked about troubleshooting or "how to fix" questions
- Be specific and reference ticket IDs when providing solutions
- If multiple similar issues exist, mention the most relevant ones
- Admit when you don't have enough information
- Be concise but thorough in your responses
- When appropriate, use multiple tools to gather complete information
- Before each tool call, provide a short public rationale in content using this exact prefix:
    "Decision: <one sentence explaining why this tool is needed>"

Remember: Your primary value is retrieving and applying solutions from past tickets!"""),
        HumanMessage(content=query)
    ]

    for i in range(max_iterations):
        response = llm_with_tools.invoke(messages)
        messages.append(response)

        # If the model produced no tool calls, treat content as final answer.
        if not response.tool_calls:
            # No more tool calls, return the response
            return response.content

        # Print model-provided public rationale (not hidden chain-of-thought).
        decision_trace = (response.content or "").strip()
        if decision_trace:
            print(f"\n🧭 {decision_trace}")

        # Execute each requested tool exactly as the model specified.
        for tool_call in response.tool_calls:
            tool_name = tool_call["name"]
            tool_input = tool_call["args"].get("input", "")

            print(f"\n🔧 Calling tool: {tool_name}")
            print(f"   Input: {tool_input}")

            # Resolve tool by name from the registered tool list.
            # This explicit lookup keeps control in application code (safer than eval).
            tool_output = None
            for tool in tools:
                if tool.name == tool_name:
                    tool_output = tool.func(tool_input)
                    break

            if tool_output is None:
                tool_output = f"Error: Tool {tool_name} not found"

            print(f"   Output: {tool_output[:200]}...")

            # Feed tool output back to model in the expected ToolMessage format.
            # The `tool_call_id` links this output to the originating request.
            messages.append(ToolMessage(
                content=tool_output,
                tool_call_id=tool_call["id"]
            ))

    return "Maximum iterations reached. Could not complete the task."

print("\n✓ Agent ready!")


✓ Agent ready!


In [17]:
print("\n" + "="*80)
print("PART 3: Specific Ticket Lookup")
print("="*80)

query2 = "Show me details of ticket TICK-005"
print(f"\nQuery: '{query2}'")
print("\nAgent will:")
print("1. Recognize this asks for a specific ticket")
print("2. Choose the GetTicketByID tool")
print("3. Return the exact ticket\n")
print("-" * 80)

response2 = run_agent(query2)
print("\n" + "-" * 80)
print("FINAL ANSWER:")
print(response2)

# ============================================================================
# PART 4: Category Filtering
# ============================================================================
print("\n" + "="*80)
print("PART 4: Category-Based Search")
print("="*80)

query3 = "What payment-related issues have we seen?"
print(f"\nQuery: '{query3}'")
print("\nAgent will:")
print("1. Identify this asks about a category of issues")
print("2. Choose the SearchByCategory tool")
print("3. Show all payment tickets\n")
print("-" * 80)

response3 = run_agent(query3)
print("\n" + "-" * 80)
print("FINAL ANSWER:")
print(response3)



PART 3: Specific Ticket Lookup

Query: 'Show me details of ticket TICK-005'

Agent will:
1. Recognize this asks for a specific ticket
2. Choose the GetTicketByID tool
3. Return the exact ticket

--------------------------------------------------------------------------------

🔧 Calling tool: GetTicketByID
   Input: TICK-005
   Output: Ticket ID: TICK-005
Title: Memory leak in background worker process
Description: Background job processor showing steady memory increase over time. Process killed by OS after reaching 8GB memory usage...

--------------------------------------------------------------------------------
FINAL ANSWER:
Here are the details for ticket **TICK-005**:

- **Title**: Memory leak in background worker process
- **Description**: The background job processor was showing a steady increase in memory usage over time, leading to the process being killed by the OS after reaching 8GB. A daily restart was required to maintain functionality.
- **Resolution**: Memory profiling